# Cleanup: Delete Workshop Resources

Deletes the resources created in **Module 6** (production deployment) and **Module 7** (long-term memory).

## How this notebook decides what to delete

Deletion is scoped **by resource tag**, never by name prefix:

    WorkshopResource=stop-ai-agent-hallucinations

A resource is deleted only if it carries that tag. Anything else in your account is left alone,
including resources whose names look like the workshop's.

This matters. An earlier version of this teardown matched IAM roles with
`role["RoleName"].startswith("AmazonBedrockAgentCoreSDKCodeBuild")` across the whole account. IAM is
global, so it reached every region and deleted five roles that this workshop never created. Name
prefixes describe what a resource is *called*; tags record who *owns* it. Only the second one is
safe to delete on.

## Two consequences worth understanding

1. **A resource that exists under a workshop name but carries no workshop tag is not deleted.** It is
   reported as `UNTAGGED_BLOCKED` and this notebook fails. That is deliberate: the resource is either
   someone else's, or it came from a deployment that forgot to tag, and neither is something a script
   should decide to delete for you.
2. **Failures are loud.** Nothing here swallows an exception. The previous version hid a `KeyError`
   behind a bare `except` and printed a clean bill of health while leaking a billable AgentCore
   Memory resource.

Run the dry run first, read the plan, then execute.

In [ ]:
import sys
from pathlib import Path

# Locate workshop_cleanup.py whether this notebook is run from 08-cleanup/ or
# from the repo root. Fails loudly rather than importing something unexpected.
candidates = [Path.cwd(), Path.cwd() / "08-cleanup", Path.cwd().parent / "08-cleanup"]
module_dir = next((p for p in candidates if (p / "workshop_cleanup.py").is_file()), None)
if module_dir is None:
    raise FileNotFoundError(
        "workshop_cleanup.py not found. Run this notebook from 08-cleanup/ or the repo root. "
        f"Looked in: {[str(p) for p in candidates]}"
    )
sys.path.insert(0, str(module_dir))

import workshop_cleanup as wc

clients = wc.Clients.build()

print(f"module:   {module_dir}")
print(f"region:   {wc.REGION}")
print(f"tag gate: {wc.WORKSHOP_TAG_KEY}={wc.WORKSHOP_TAG_VALUE}")

---
## Step 1: Dry run

Read-only. Lists every candidate, whether it is selected, and why. Nothing is deleted.

In [ ]:
plan = wc.build_plan(clients)
wc.print_plan(plan, dry_run=True, region=clients.region)

---
## Step 2: Review the plan

Before running the next cell, confirm that every line marked `DELETE` is a resource you created in
Module 6 or 7. Lines marked `SKIP` are either absent or untagged, and will not be touched.

In [ ]:
for candidate in plan:
    if candidate.will_delete:
        print(candidate.describe())

---
## Step 3: Execute

Deletes the selected resources. Raises if anything failed or if any untagged workshop resource was
found, so a failed teardown cannot be mistaken for a successful one.

In [ ]:
exit_code = wc.run(clients, dry_run=False)

if exit_code != 0:
    raise RuntimeError(
        "Cleanup did not complete. Read the BLOCKED and FAILURES output above — "
        "some resources may still exist and may still be billing."
    )

print("Teardown completed with no failures.")

---
## Step 4: Verify by listing, not by trusting this notebook

A success message is a claim, not evidence. Re-running the plan should now show every workshop
resource as `not found`.

In [ ]:
after = wc.build_plan(clients)
remaining = [c for c in after if c.selection is not wc.Selection.ABSENT]

wc.print_plan(after, dry_run=True, region=clients.region)

if remaining:
    raise RuntimeError(f"{len(remaining)} workshop resource(s) still present after cleanup")

print("Verified: no workshop resources remain.")

---
## What this does NOT delete

- **Neo4j infrastructure** — the Code Editor EC2 instance or the Central Neo4j ECS stack from
  Module 1. Those come from CloudFormation: delete via AWS Console → CloudFormation → Delete Stack.
- **CloudWatch log groups** — retained so you can review the run. Delete them by hand if you want.
- **Anything untagged.** By design. See the note at the top.

## Command line equivalent

    python workshop_cleanup.py             # dry run by default: nothing is deleted
    python workshop_cleanup.py --dry-run   # the same read-only run, made explicit
    python workshop_cleanup.py --yes       # execute it and delete the tagged resources

The default is a dry run. Deletion happens only when you pass `--yes`. All three exit non-zero if the
teardown is incomplete, so they are safe to use in a script.